# Notebook 5 - Leakage-Safe Feature Engineering

## Goal

Build features available at order time, fit preprocessing objects on the training split only, apply the fitted objects unchanged to validation and test, and save the matrices, transformer, and feature list.

Excluded leakage columns include actual delivery timestamps, carrier timestamps, delivery delay, reviews, and order status.

In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
FIGURES = ROOT / "figures"
REPORTS = ROOT / "reports"
for folder in [ARTIFACTS, FIGURES, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)
print(f"Task root: {ROOT.resolve()}")

Task root: C:\Users\moath\Desktop\MLOPS\MLOps_Rand_Salem\MLOps-Qafza-2026\Tasks\Task-02


In [2]:
import numpy as np
import pandas as pd
import joblib
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

paths = {name: ARTIFACTS / f"03_{name}.parquet" for name in ["train", "validation", "test"]}
assert all(path.exists() for path in paths.values()), "Run Notebook 3 first."
splits = {name: pd.read_parquet(path) for name, path in paths.items()}
print({name: frame.shape for name, frame in splits.items()})

{'train': (67529, 39), 'validation': (14470, 39), 'test': (14471, 39)}


## 1. Create prediction-time features

In [3]:
def build_feature_frame(frame):
    out = frame.copy()
    purchase = pd.to_datetime(out["order_purchase_timestamp"], errors="coerce")
    estimated = pd.to_datetime(out["order_estimated_delivery_date"], errors="coerce")
    out["purchase_month"] = purchase.dt.month
    out["purchase_day_of_week"] = purchase.dt.dayofweek
    out["purchase_hour"] = purchase.dt.hour
    out["purchase_is_weekend"] = purchase.dt.dayofweek.isin([5, 6]).astype("int8")
    out["estimated_delivery_days"] = (estimated - purchase).dt.total_seconds() / 86400
    return out

feature_splits = {name: build_feature_frame(frame) for name, frame in splits.items()}

numeric_features = [
    "customer_zip_code_prefix", "item_count", "product_count", "seller_count",
    "price_total", "price_mean", "freight_total", "freight_mean",
    "product_weight_mean", "product_length_mean", "product_height_mean",
    "product_width_mean", "product_photos_mean", "seller_lat_mean",
    "seller_lng_mean", "seller_state_nunique", "payment_count",
    "payment_value_total", "payment_installments_max", "payment_type_count",
    "customer_lat", "customer_lng", "customer_seller_distance_km",
    "purchase_month", "purchase_day_of_week", "purchase_hour",
    "purchase_is_weekend", "estimated_delivery_days",
]
categorical_features = [
    "customer_state", "primary_seller_state", "primary_category", "dominant_payment_type",
]

required = set(numeric_features + categorical_features + ["order_id", "is_late"])
missing = required - set(feature_splits["train"].columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

## 2. Fit preprocessing on training data only

In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler(with_mean=False)),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=True)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], sparse_threshold=1.0)

X_train_raw = feature_splits["train"][numeric_features + categorical_features]
X_train = preprocessor.fit_transform(X_train_raw)
X_validation = preprocessor.transform(feature_splits["validation"][numeric_features + categorical_features])
X_test = preprocessor.transform(feature_splits["test"][numeric_features + categorical_features])

assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]
print("Feature matrix shapes:", X_train.shape, X_validation.shape, X_test.shape)

Feature matrix shapes: (67529, 157) (14470, 157) (14471, 157)


## 3. Save every fitted object and output artifact

In [5]:
matrices = {"train": X_train, "validation": X_validation, "test": X_test}
for name, matrix in matrices.items():
    sparse.save_npz(ARTIFACTS / f"05_X_{name}.npz", sparse.csr_matrix(matrix))
    np.save(ARTIFACTS / f"05_y_{name}.npy", feature_splits[name]["is_late"].to_numpy(dtype="int8"))
    feature_splits[name][["order_id", "order_purchase_timestamp"]].to_csv(
        ARTIFACTS / f"05_ids_{name}.csv", index=False
    )

joblib.dump(preprocessor, ARTIFACTS / "05_preprocessor.joblib")
feature_names = preprocessor.get_feature_names_out().tolist()
(ARTIFACTS / "05_feature_names.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")

manifest = {
    "fit_split": "train only",
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "output_feature_count": len(feature_names),
    "matrix_shapes": {name: list(matrix.shape) for name, matrix in matrices.items()},
    "leakage_columns_excluded": [
        "order_status", "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "delivery_delay_days", "reviews",
    ],
}
(REPORTS / "05_feature_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Saved {len(feature_names)} final features and the fitted preprocessor.")
manifest

Saved 157 final features and the fitted preprocessor.


{'fit_split': 'train only',
 'numeric_features': ['customer_zip_code_prefix',
  'item_count',
  'product_count',
  'seller_count',
  'price_total',
  'price_mean',
  'freight_total',
  'freight_mean',
  'product_weight_mean',
  'product_length_mean',
  'product_height_mean',
  'product_width_mean',
  'product_photos_mean',
  'seller_lat_mean',
  'seller_lng_mean',
  'seller_state_nunique',
  'payment_count',
  'payment_value_total',
  'payment_installments_max',
  'payment_type_count',
  'customer_lat',
  'customer_lng',
  'customer_seller_distance_km',
  'purchase_month',
  'purchase_day_of_week',
  'purchase_hour',
  'purchase_is_weekend',
  'estimated_delivery_days'],
 'categorical_features': ['customer_state',
  'primary_seller_state',
  'primary_category',
  'dominant_payment_type'],
 'output_feature_count': 157,
 'matrix_shapes': {'train': [67529, 157],
  'validation': [14470, 157],
  'test': [14471, 157]},
 'leakage_columns_excluded': ['order_status',
  'order_approved_at',
  'o